In [ ]:
!pip install sae-lens transformer-lens


In [1]:
!pip install "numpy<2.0.0"

In [ ]:
import torch
import json
import gc
from datasets import load_dataset
from tqdm import tqdm
from transformer_lens import HookedTransformer
from transformer_lens.past_key_value_caching import HookedTransformerKeyValueCache
from huggingface_hub import login
from sae_lens import SAE
# --- Setup & Initialization ---
HF_TOKEN = "" # Add your token here
login(token=HF_TOKEN)
print("Environment setup initialized.")

gc.collect()
torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"
n_gpus = torch.cuda.device_count()
print(f"Detected {n_gpus} GPUs. Using device: {device}")

target_layer = 12
early_layer = 8 # Chosen for the generic/baseline injection
target_neuron_idx = 8526 

print("\n--- Loading Gemma-2-2B ---")
model = HookedTransformer.from_pretrained_no_processing(
    "gemma-2-2b", 
    device="cuda", 
    n_devices=2,              
    dtype=torch.bfloat16,
    center_unembed=False,          
    center_writing_weights=False  
)

print("Fixing tied embeddings for multi-GPU...")
# Identify where the final layers were placed (should be cuda:1)
last_device = next(model.blocks[-1].parameters()).device

# Clone and force the unembedding weights onto the second GPU
model.unembed.W_U = torch.nn.Parameter(model.unembed.W_U.clone().to(last_device))
if model.unembed.b_U is not None:
    model.unembed.b_U = torch.nn.Parameter(model.unembed.b_U.clone().to(last_device))

print(f"Model successfully distributed. Unembed is on {last_device}")

# --- Phase 1: Compute the Generic Mean Vector ---
print("\n--- PART 1: Computing Generic Mean Vector ---")
dataset_baseline = load_dataset("NeelNanda/pile-10k", split="train[:100]")
# Force the output into a standard Python list of strings
baseline_prompts = [str(text) for text in dataset_baseline["text"]]

# Tokenize and pad to a uniform length to process as a batch
baseline_tokens = model.to_tokens(baseline_prompts, truncate=True, padding_side="left")
# Take just the last 32 tokens to keep memory usage manageable
baseline_tokens = baseline_tokens[:, -32:] 

hook_early_resid = f"blocks.{early_layer}.hook_resid_pre"

with torch.no_grad():
    _, cache = model.run_with_cache(baseline_tokens, names_filter=[hook_early_resid])
    # Slice out the last token of every sequence: tensor[:, -1, :]
    early_resid_acts = cache[hook_early_resid][:, -1, :] 
    # Collapse the batch dimension
    Mean_Vector = early_resid_acts.mean(dim=0).detach().clone()

print(f"Mean Vector shape: {Mean_Vector.shape}")

# --- Phase 2: Gradient-Based Search ---
print("\n--- PART 2: Gradient-Based Search for Optimized Vector ---")
torch.set_grad_enabled(True) # Enable gradients for this phase

# Initialize learnable vector with the Mean_Vector
X = Mean_Vector.clone().requires_grad_(True)
optimizer = torch.optim.Adam([X], lr=0.05)
hook_late_mlp = f"blocks.{target_layer}.mlp.hook_post"

dummy_prompt = "<pad> <pad> <pad> x"
dummy_tokens = model.to_tokens(dummy_prompt)
opt_target_pos = dummy_tokens.shape[1] - 1

def inject_learnable_x_hook(resid, hook):
    # Dynamically move X to whatever GPU this specific layer is on
    resid[:, opt_target_pos, :] = X.to(resid.dtype).to(resid.device)
    return resid

print(f"Optimizing to maximize Neuron {target_neuron_idx} in Layer {target_layer}...")
num_steps = 50

for step in range(num_steps):
    optimizer.zero_grad()
    
    # 1. Create an empty list to capture the loss from inside the hook
    loss_list = [] 
    
    def calculate_loss_hook(mlp_post, hook):
        # 2. No 'nonlocal' needed. Calculate the loss and append it to the list.
        current_loss = -mlp_post[0, opt_target_pos, target_neuron_idx]
        loss_list.append(current_loss)
        return mlp_post

    model.run_with_hooks(
        dummy_tokens, 
        fwd_hooks=[
            (hook_early_resid, inject_learnable_x_hook),
            (hook_late_mlp, calculate_loss_hook)
        ]
    )
    
    # 3. Pull the loss out of the list and backpropagate
    loss = loss_list[0]
    loss.backward()
    optimizer.step()
    
    if step % 10 == 0:
        print(f" Step {step} | Activation (Negative Loss): {-loss.item():.4f}")

Optimized_Vector = X.detach().clone()
torch.set_grad_enabled(False) # Turn off gradients for inference
print("Optimization complete. Optimized_Vector frozen.")

# --- Phase 2.5: SAE Verification of Optimized Vector ---
print(f"\n--- PART 2.5: SAE Verification of Optimized Vector (Layer {early_layer}) ---")

# 1. Load the SAE for the Early Layer where the vector lives
sae_id = f"layer_{early_layer}/width_16k/canonical"
print(f"Loading SAE: {sae_id}...")
sae = SAE.from_pretrained(
    release="gemma-scope-2b-pt-res-canonical",
    sae_id=sae_id,
    device=device # Load onto your primary GPU
)

# 2. Format the Optimized_Vector for the SAE
# We add a batch dimension and match the SAE's expected precision
opt_vec_for_sae = Optimized_Vector.unsqueeze(0).to(sae.device).to(sae.W_enc.dtype)

# 3. Encode the vector through the SAE to get feature activations
with torch.no_grad():
    feature_acts = sae.encode(opt_vec_for_sae)
    
    # Grab the top 5 most active concepts
    top_acts, top_indices = feature_acts[0].topk(5)

print("\nTop 5 Activated SAE Features in the Optimized_Vector:")
for act, idx in zip(top_acts, top_indices):
    if act.item() > 0:
        print(f" - Feature_{idx.item()}: Activation = {act.item():.4f}")
        print(f"   Neuronpedia: https://neuronpedia.org/gemma-2-2b/{early_layer}-gemmascope-res-16k/{idx.item()}")

# Clean up SAE to free VRAM before Contrastive Decoding
del sae, opt_vec_for_sae, feature_acts
gc.collect()
torch.cuda.empty_cache()


# --- Phase 3: The Patchscope Setup ---
print("\n--- PART 3: Patchscope Setup ---")
target_prompt = "A detailed description of x is that"
target_tokens = model.to_tokens(target_prompt)
target_str_tokens = model.to_str_tokens(target_prompt)

try:
    target_pos = next(i for i, t in enumerate(target_str_tokens) if 'x' in t.lower())
except StopIteration:
    target_pos = target_tokens.shape[1] - 3 

print(f"Prompt: '{target_prompt}'")
print(f"Injection point: Token '{target_str_tokens[target_pos]}' at position {target_pos}")

# --- Phase 4: Contrastive Decoding with KV Cache ---
print("\n--- PART 4: Contrastive Decoding Loop ---")

alpha = 1.5 # Contrastive weighting factor
max_new_tokens = 20

# Initialize KV Caches for Run A (Optimized) and Run B (Mean)
kv_cache_A = HookedTransformerKeyValueCache.init_cache(model.cfg, device, batch_size=1)
kv_cache_B = HookedTransformerKeyValueCache.init_cache(model.cfg, device, batch_size=1)

def prefill_injection_hook(vector_to_inject):
    def hook_fn(resid, hook):
        if resid.shape[1] > 1 and resid.shape[1] > target_pos:
            # Dynamically move the injected vector to the layer's GPU
            resid[:, target_pos, :] = vector_to_inject.to(resid.dtype).to(resid.device)
        return resid
    return hook_fn

generated_tokens = []
current_tokens = target_tokens.clone()

# Prefill Pass (Generating token 1)
# Run A (The Signal)
# --- ADD THIS: List to capture the activation ---
patchscope_activation = []

def capture_target_activation_hook(mlp_post, hook):
    # Capture the activation of the target neuron at the target position
    act = mlp_post[0, target_pos, target_neuron_idx].item()
    patchscope_activation.append(act)
    return mlp_post

# Prefill Pass (Generating token 1)
# Run A (The Signal)
logits_A = model.run_with_hooks(
    current_tokens, 
    past_kv_cache=kv_cache_A,
    fwd_hooks=[
        (hook_early_resid, prefill_injection_hook(Optimized_Vector)),
        (f"blocks.{target_layer}.mlp.hook_post", capture_target_activation_hook) # Added hook here
    ]
)
L_act = logits_A[0, -1, :]

print(f"Target Neuron {target_neuron_idx} activation during Patchscope prefill: {patchscope_activation[0]:.4f}")

# Run B (The Noise)
logits_B = model.run_with_hooks(
    current_tokens, 
    past_kv_cache=kv_cache_B,
    fwd_hooks=[(hook_early_resid, prefill_injection_hook(Mean_Vector))]
)
L_mean = logits_B[0, -1, :]

# Calculate & Sample (Using Joseph's formula)
Final_Logits = L_act + alpha * (L_act - L_mean)
next_token = Final_Logits.argmax(dim=-1).unsqueeze(0).unsqueeze(0)
generated_tokens.append(next_token.item())

# CHANGE: Send token back to GPU 0 for the start of the model
current_tokens = next_token.to("cuda:0")

# Autoregressive Loop (Tokens 2 through N)
for _ in tqdm(range(max_new_tokens - 1), desc="Generating"):
    # Run A (Bypasses hook automatically since seq_len == 1)
    logits_A = model.run_with_hooks(
        current_tokens, 
        past_kv_cache=kv_cache_A,
        fwd_hooks=[(hook_early_resid, prefill_injection_hook(Optimized_Vector))]
    )
    L_act = logits_A[0, -1, :]
    
    # Run B 
    logits_B = model.run_with_hooks(
        current_tokens, 
        past_kv_cache=kv_cache_B,
        fwd_hooks=[(hook_early_resid, prefill_injection_hook(Mean_Vector))]
    )
    L_mean = logits_B[0, -1, :]
    
    Final_Logits = L_act + alpha * (L_act - L_mean)
    next_token = Final_Logits.argmax(dim=-1).unsqueeze(0).unsqueeze(0)
    
    generated_tokens.append(next_token.item())
    
    # CHANGE: Send token back to GPU 0 for the start of the model
    current_tokens = next_token.to("cuda:0")

# --- Phase 5: Verification and Metrics ---
print("\n--- PART 5: Output Analysis ---")
final_output = model.tokenizer.decode(generated_tokens)
print(f"Targeting Layer {target_layer}, Neuron {target_neuron_idx}")
print(f"Contrastive Alpha (\u03b1): {alpha}") # Used unicode for terminal compatibility
print(f"\nGenerated Output:\n{target_prompt} {final_output}")

2026-03-10 05:33:16.743184: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773120797.186769     132 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773120797.297336     132 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773120798.250232     132 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773120798.250280     132 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773120798.250284     132 computation_placer.cc:177] computation placer alr

Environment setup initialized.
Detected 2 GPUs. Using device: cuda

--- Loading Gemma-2-2B ---


config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

Mean Vector shape: torch.Size([2304])

--- PART 2: Gradient-Based Search for Optimized Vector ---
Optimizing to maximize Neuron 8526 in Layer 12...
 Step 0 | Activation (Negative Loss): 4.8125
 Step 10 | Activation (Negative Loss): 18.8750
 Step 20 | Activation (Negative Loss): 30.2500
 Step 30 | Activation (Negative Loss): 40.5000
 Step 40 | Activation (Negative Loss): 50.0000
Optimization complete. Optimized_Vector frozen.

--- PART 2.5: SAE Verification of Optimized Vector (Layer 8) ---
Loading SAE: layer_8/width_16k/canonical...


layer_8/width_16k/average_l0_71/params.n(…):   0%|          | 0.00/302M [00:00<?, ?B/s]


Top 5 Activated SAE Features in the Optimized_Vector:
 - Feature_302: Activation = 41.6082
   Neuronpedia: https://neuronpedia.org/gemma-2-2b/8-gemmascope-res-16k/302
 - Feature_3124: Activation = 12.1132
   Neuronpedia: https://neuronpedia.org/gemma-2-2b/8-gemmascope-res-16k/3124
 - Feature_15075: Activation = 9.4050
   Neuronpedia: https://neuronpedia.org/gemma-2-2b/8-gemmascope-res-16k/15075
 - Feature_10524: Activation = 8.2802
   Neuronpedia: https://neuronpedia.org/gemma-2-2b/8-gemmascope-res-16k/10524
 - Feature_4526: Activation = 6.4116
   Neuronpedia: https://neuronpedia.org/gemma-2-2b/8-gemmascope-res-16k/4526

--- PART 3: Patchscope Setup ---
Prompt: 'A detailed description of x is that'
Injection point: Token ' x' at position 5

--- PART 4: Contrastive Decoding Loop ---
Target Neuron 8526 activation during Patchscope prefill: 56.0000


Generating: 100%|██████████| 19/19 [00:04<00:00,  4.66it/s]


--- PART 5: Output Analysis ---
Targeting Layer 12, Neuron 8526
Contrastive Alpha (α): 1.5

Generated Output:
A detailed description of x is that  it is a number that is not a real number. So, it is a number that is not
